# 🌊 AquaRoute AI: Flood Disaster Road Criticality & Evacuation Optimization
### *A 4-Step Mathematical & Network Optimization Pipeline for Emergency Operations Centers (EOC)*

---

## 🎯 Objective
During catastrophic urban floods (e.g. Mumbai Monsoon Deluge, Jakarta Coastal Rob Flood, Houston Hurricane Harvey), emergency managers face critical questions:
1. **Which roads are submerged, and how fast will vehicles stall?**
2. **Where will panic evacuation traffic cause massive network gridlock?**
3. **Which bridges and arterial intersections form single-point-of-failure bottlenecks?**
4. **Where should emergency pumps, sandbag barriers, and contraflow corridors be deployed first to save the most lives?**

This notebook walks through the full 4-step Python simulation pipeline.

## 📐 Mathematical Formulation

### Step 1: Flood Inundation & Vehicle Speed Degradation
Water depth $h(x,y)$ is calculated combining pluvial rainfall runoff, fluvial/coastal surge, and Digital Elevation Model (DEM) attenuation:
$$h(x,y) = \max\left(0, h_{base}(R) + S_{surge} \cdot e^{-d_{water}/\lambda} \cdot \text{ElevAdj}(z)\right)$$

Effective vehicle travel speed $v_{eff}$ degrades non-linearly with water depth $h$:
$$v_{eff}(e) = v_{max}(e) \cdot \max\left(0, 1 - \left(\frac{h_e}{h_{crit}}\right)^2\right)$$
where $h_{crit} = 0.35\text{ m}$ (standard vehicle stall threshold) and $0.60\text{ m}$ for emergency 4x4s.

---

### Step 2: Dynamic Evacuation Flow Field
Evacuation traffic $F_e$ is assigned across capacity-constrained shortest paths minimizing total flood impedance:
$$Z(e) = t_{travel}(e) + \alpha \cdot (h_e)^{\beta}$$

---

### Step 3: Capacity vs Volume Bottleneck Detection
The Capacity-to-Volume Ratio ($CVR$) identifies over-saturated road links:
$$CVR(e) = \frac{F_e}{\text{Cap}_e}$$
Network cut-off risk evaluates degree degradation when flooded links fail:
$$I_{cutoff}(v) = \frac{\deg(v) - \deg_{dry}(v)}{\deg(v)}$$

---

### Step 4: Multi-Factor Road Criticality Score (RCS)
$$RCS(e) = w_1 \cdot C_B(e) + w_2 \cdot \min(2.5, CVR(e)) + w_3 \cdot V(e) + w_4 \cdot I_{cutoff}(e)$$
normalized to a 0–100 index.

In [ ]:
import os
import sys
import json
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx

# Ensure project root is on sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.config import CITIES_CONFIG
from src.osm_fetcher import NetworkFetcher
from src.step1_hazard import HazardSimulator
from src.step2_flow_field import FlowFieldSimulator
from src.step3_bottlenecks import BottleneckDetector
from src.step4_road_scoring import RoadScoringOptimizer

print("✅ AquaRoute AI Modules successfully imported!")

## 🚀 Running the 4-Step Simulation for Mumbai

In [ ]:
# Load Mumbai Configuration
mumbai_config = CITIES_CONFIG["mumbai"]
scenario_severe = mumbai_config["scenarios"]["severe"]

# Construct Network Graph
fetcher = NetworkFetcher()
G = fetcher.get_or_build_graph(mumbai_config, force_synthetic=True)
print(f"Network constructed: {G.number_of_nodes()} nodes, {G.number_of_edges()} directional edges.")

# Step 1: Hazard Simulation
hazard_sim = HazardSimulator(mumbai_config)
G = hazard_sim.apply_hazard_to_graph(G, scenario_severe)

# Step 2: Flow Field
flow_sim = FlowFieldSimulator(mumbai_config)
origins = flow_sim.generate_evacuation_demand(G)
G, routes = flow_sim.simulate_evacuation_flows(G, origins)

# Step 3: Bottlenecks
bottleneck_detector = BottleneckDetector()
G, edge_bnecks, node_chokes = bottleneck_detector.analyze_bottlenecks(G)

# Step 4: Road Criticality Scoring
optimizer = RoadScoringOptimizer()
G, scored_roads = optimizer.compute_road_criticality(G)
interventions = optimizer.generate_intervention_plan(scored_roads, node_chokes)

print(f"Simulation finished! Identified {len(interventions['priority_interventions'])} top emergency interventions.")

## 📊 Data Visualization: Critical Corridors & Bottleneck Pressure

In [ ]:
# Top 10 Scored Roads by Criticality (RCS)
top_roads = scored_roads[:10]
road_names = [r["name"][:22] for r in top_roads]
scores = [r["criticality_score"] for r in top_roads]
depths = [r["flood_depth_m"] for r in top_roads]
flows = [r["flow_volume"] for r in top_roads]

plt.style.use('dark_background')
fig, ax1 = plt.subplots(figsize=(12, 6))

color = '#38bdf8'
ax1.set_xlabel('Road Corridor', fontsize=12, fontweight='bold')
ax1.set_ylabel('Road Criticality Score (RCS)', color=color, fontsize=12, fontweight='bold')
bars = ax1.bar(road_names, scores, color=color, alpha=0.8, label='RCS Score')
ax1.tick_params(axis='y', labelcolor=color)
plt.xticks(rotation=45, ha='right')

ax2 = ax1.twinx()
color = '#ef4444'
ax2.set_ylabel('Inundation Depth (m)', color=color, fontsize=12, fontweight='bold')
ax2.plot(road_names, depths, color=color, marker='o', linewidth=2.5, label='Water Depth')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Mumbai Flood Simulation: Critical Lifeline Road Ranking & Water Inundation', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 🚒 Actionable Emergency Interventions Queue

In [ ]:
import pandas as pd
plan_data = []
for it in interventions["priority_interventions"]:
    plan_data.append({
        "Rank": f"#{it['rank']}",
        "Corridor": it["target_road"],
        "RCS Score": it["criticality_score"],
        "Flood Depth (m)": it["flood_depth_m"],
        "Flow Pressure": it["flow_pressure"],
        "Recommended Action": it["action_summary"]
    })

for row in plan_data:
    print(f"{row['Rank']} | {row['Corridor']:<35} | RCS: {row['RCS Score']} | Water: {row['Flood Depth (m)']}m | Action: {row['Recommended Action']}")

## 🏆 Conclusion & Real-World EOC Impact
- **Proactive Resource Dispatch**: Rather than waiting for stalled vehicles, EOC commanders can pre-deploy high-capacity mobile pumps and rapid sandbag barriers to the exact Tier-1 corridors identified.
- **Multi-City Resilience**: Easily generalizable to Mumbai, Jakarta, Houston, or any coastal/fluvial metropolis worldwide.
- **GitHub Pages Demo**: Open `web/index.html` in your browser for the full interactive visual command center.